# LOG635 – Laboratoire 1 : Parcourir le monde

## Imports

In [ ]:
# Chemin absolu de ce script sur le robot :
# /home/pi/

import glob      
import os
import time      
from datetime import datetime 

import cv2       # OpenCV : capture des images de la caméra USB et enregistrement en JPEG

# SDK du Raspbot : accès à la carte d'extension (capteurs, servo, buzzer, LED, etc...)
from Raspbot_Lib import Raspbot

# Bibliothèque Yahboom des roues pour la vitesse de chacune des 4 roues
from McLumk_Wheel_Sports import move_forward, rotate_left, rotate_right, stop_robot

## Constantes et objet robot

In [ ]:
NEAR_DISTANCE = 150          # distance pour que le robot détecte un objet
FAR_DISTANCE = 425           # distance pour que le robot considère l'objet retiré du chemin
OBSTACLE_POLL_INTERVAL = 0.5 # vérifications de la distance pendant l'attente devant un objet en seconde
LAST_MOVE = "forward"        # mémoire du dernier mouvement : sert à retrouver la ligne si elle est perdue

motion_speed = 5             # vitesse de base des déplacements

bot = Raspbot()             

## Initialisation et arrêt des systèmes

In [ ]:
def _init_systems():
    """Initialize the robot's systems."""
    bot.Ctrl_Ulatist_Switch(1)  # allume le capteur ultrason (éteint par défaut)
    bot.Ctrl_Servo(2, 0)        # incline la caméra vers le bas pour prendre des photos
    time.sleep(0.1)             # laisse à la carte le temps d'appliquer les deux commandes


def _close_systems():
    """Close the robot's systems."""
    bot.Ctrl_Ulatist_Switch(0)  # éteint le capteur ultrason
    bot.Ctrl_BEEP_Switch(0)     # coupe le buzzer, au cas où il serait resté actif
    bot.Ctrl_WQ2812_ALL(0, 6)   # éteint la barre de LED
    stop_robot()                # met les 4 moteurs à l'arrêt
    time.sleep(0.1)

## Lecture du capteur ultrason

In [ ]:
def read_ultrasonic_distance():
    """Read the distance reported by the ultrasonic sensor.

    Combine the high byte from register ``0x1B`` and the low byte from
    register ``0x1A``. For example, ``read_ultrasonic_distance()`` returns
    the current obstacle distance in millimetres.

    Returns:
        The measured distance in millimetres.
    """

    distance_high = int(bot.read_data_array(0x1B, 1)[0])  
    distance_low = int(bot.read_data_array(0x1A, 1)[0])   

    return (distance_high << 8) | distance_low

## Signal sonore (buzzer)

In [ ]:
def sound_buzzer(times=1, duration=0.1, interval=0.1):
    """Sound the buzzer for a given duration in seconds.

    Uses the bot.Ctrl_BEEP_Switch attribute to control the buzzer state.

    Args:
        times: Number of times to sound the buzzer.
        duration: Time in seconds to keep the buzzer on.
        interval: Time in seconds between buzzer on and off states.
    """
    for _ in range(times):          # répète le bip le nombre de fois demandé
        bot.Ctrl_BEEP_Switch(1)     # allume le buzzer
        time.sleep(duration)        # durée du bip
        bot.Ctrl_BEEP_Switch(0)     # éteint le buzzer
        time.sleep(interval)        

## Signal lumineux (barre de LED)

In [ ]:
def blink_leds(times=3, interval=0.2):
    """Blink the LEDs for a given number of times.

    Uses the bot.Ctrl_WQ2812_ALL attribute to control the
    LED state.

    Args:
        times: Number of times to blink the LEDs.
        interval: Time in seconds between LED on and off states.
    """
    for _ in range(times):
        bot.Ctrl_WQ2812_ALL(1, 6)  # allume la LED
        time.sleep(interval)
        bot.Ctrl_WQ2812_ALL(0, 6)  # éteint toutes la LED
        time.sleep(interval)

## Ouverture de la caméra

In [ ]:
def open_camera():
    """Open the first video device that produces a frame.

    Probe numbered Linux video devices in order because Raspberry Pi 5 can
    expose processing devices alongside the USB camera. For example,
    ``camera, device = open_camera()`` opens a usable capture device.

    Returns:
        A ``(camera, device_path)`` pair, or ``(None, None)`` if no device
        can provide a frame. The caller must release a returned camera.
    """
    
    devices = [
        path for path in glob.glob("/dev/video*") if path[len("/dev/video") :].isdigit()
    ]
    devices.sort(key=lambda path: int(path[len("/dev/video") :])) 
    for device in devices:
        camera = cv2.VideoCapture(device, cv2.CAP_V4L2)  
        usable = False
        try:
            if camera.isOpened():
                ok, frame = camera.read()      
                usable = ok and frame is not None  
                if usable:
                    return camera, device       
        finally:
            if not usable:
                camera.release()                 
    return None, None                            

## Prise de photos

In [ ]:
def take_photos(count=1, interval=0.5, directory="photos-eqK"):
    """Capture JPEG photos with the first available USB camera.

    Save photos in a local directory and return only successfully written
    files. For example, ``take_photos(count=3, interval=1)`` captures three
    photos one second apart.

    Args:
        count: Number of photos to attempt.
        interval: Seconds to wait between captures.
        directory: Destination directory, created if needed.

    Returns:
        A list of saved JPEG file paths; empty if no camera is available.

    Raises:
        RuntimeError: No usable camera is found.
    """
    camera, device = open_camera()
    if camera is None:
        raise RuntimeError("No usable camera found.")  

    print(f"Camera: {device}")
    files = []
    try:
        os.makedirs(directory, exist_ok=True)              # crée le dossier des photos (photos-eqK)
        camera.set(cv2.CAP_PROP_FRAME_WIDTH, 640)          
        camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

        for index in range(count):
            ok, frame = camera.read()                      
            if not ok or frame is None:
                print(f"Error: photo {index + 1} could not be captured.")
            else:
                # Pour des noms uniques et un classement chronologique
                name = (
                    datetime.now().strftime("%Y%m%d_%H%M%S_%f") + f"_{index + 1:03}.jpg"
                )
                path = os.path.join(directory, name)
                if cv2.imwrite(path, frame):               # enregistre l'image sur le disque
                    files.append(path)
                    print(f"Photo saved: {path}")
                else:
                    print(f"Error: photo {index + 1} could not be saved to {path}.")

            if index < count - 1:                          
                time.sleep(interval)
    finally:
        camera.release()  # libère toujours la caméra, sinon elle reste bloquée pour la suite

    return files

## Suivi de la ligne noire


In [ ]:
def follow_black_line(motion_speed: int = 15):
    """Read the IR sensors and update the ongoing line-following movement.

    Register 0x0a contains four active-low sensor bits. X2, X1, X3, and X4
    correspond to the outer-left, inner-left, inner-right, and outer-right
    sensors, respectively.
    The mapping for the four sensors is as follows:
    X2 X1 X3 X4
    |  |  |  |
    L1 L2 R1 R2
    L1: Left outermost sensor
    L2: Left inner sensor
    R1: Right inner sensor
    R2: Right outermost sensor

    Example: ```follow_black_line(30)``` reads and responds once.

    Args:
        motion_speed: Base speed sent to the movement helpers.
    """
    global LAST_MOVE  # on modifie la variable globale qui mémorise le dernier sens de rotation

    # Refuse une vitesse hors de l'intervalle prévu, pour éviter un départ incontrôlé.
    if not isinstance(motion_speed, int) or motion_speed < 0 or motion_speed > 100:
        raise ValueError("motion_speed must be an integer between 0 and 100.")

    # les 4 capteurs dans un seul octet
    track = int(bot.read_data_array(0x0A, 1)[0])
    line_l1 = (track >> 2) & 0x01  # capteur extérieur gauche
    line_l2 = (track >> 3) & 0x01  # capteur intérieur gauche
    line_r1 = (track >> 1) & 0x01  # capteur intérieur droit
    line_r2 = track & 0x01         # capteur extérieur droit

    if (
        line_l1 == line_l2 == line_r1 == line_r2 == 0
    ):  
        # Les 4 capteurs sur le noir : intersection ou ligne perpendiculaire, on traverse tout droit
        decision = "1"
        LAST_MOVE = "forward"
        movement, speed = move_forward, motion_speed
    elif (
        line_l2 == 0 or line_l1 == 0
    ) and line_r2 == 0:  
        # Left sensors on black, right sensor on white, turn right
        decision = "2"
        LAST_MOVE = "right"
        movement, speed = rotate_right, int(motion_speed * 6.8)
    elif line_l1 == 0 and (
        line_r2 == 0 or line_r1 == 0
    ):  # Left sensor on white, right sensors on black, turn left
        decision = "3"
        LAST_MOVE = "left"
        movement, speed = rotate_left, int(motion_speed * 6.8)
    elif line_l1 == 0:  
        # Left outer sensor on black, turn left
        # Seul le capteur extérieur gauche voit la ligne : tourne à gauche.
        decision = "4"
        LAST_MOVE = "left"
        movement, speed = rotate_left, motion_speed
    elif line_r2 == 0:  
        # Right outer sensor on black, turn right
        # Seul le capteur extérieur droit voit la ligne : tourne à droite.
        decision = "5"
        LAST_MOVE = "right"
        movement, speed = rotate_right, motion_speed
    elif (
        line_l2 == 0 and line_r1 == 1
    ):  # Left inner sensor on black, right inner sensor on white, rotate left
        # Léger décalage vers la droite : correction douce vers la gauche
        decision = "6"
        LAST_MOVE = "left"
        movement, speed = rotate_left, int(motion_speed * 0.5)
    elif (
        line_l2 == 1 and line_r1 == 0
    ):  # Left inner sensor on white, right inner sensor on black, rotate right
        # Léger décalage vers la gauche : correction douce vers la droite.
        decision = "7"
        LAST_MOVE = "right"
        movement, speed = rotate_right, int(motion_speed * 0.5)
    elif line_l2 == 0 and line_r1 == 0:  
        # Both inner sensors on black, move forward
        # Les deux capteurs du centre sur la ligne, le robot est aligné, il avance.
        decision = "8"
        movement, speed = move_forward, int(motion_speed * 0.5)
    else:
        # Aucun capteur sur le noir, la ligne est perdue
        # Le robot pivote dans le dernier sens utilisé pour ramener la ligne sous ses capteurs.
        decision = "Line lost"
        if LAST_MOVE == "left":
            movement, speed = rotate_left, int(motion_speed * 0.5)
        elif LAST_MOVE == "right":
            movement, speed = rotate_right, int(motion_speed * 0.5)
        else:
            movement = None

    print(decision)
    print(LAST_MOVE)
    print(line_l1, line_l2, line_r1, line_r2)

    if movement is None:
        stop_robot()       
    else:
        movement(speed)     # envoie la commande choisie aux moteurs
    time.sleep(0.01)        

## Boucle principale

In [ ]:
def main():
    try:
        _init_systems()  # allume le capteur ultrason et oriente la caméra
        while True:
            follow_black_line(motion_speed)  # déplacement du robot

            distance = read_ultrasonic_distance()
            if distance > NEAR_DISTANCE:
                continue                     

            # Objet détecté donc :
            stop_robot()                     # arrête les moteurs (4 roues)
            sound_buzzer(times=2)            # buzzer pendant 2s

            # prendre la photo de l'objet
            try:
                photos = take_photos(count=1) 
                time.sleep(1)
                if not photos:
                    print("Warning: obstacle photo could not be captured.")
            except RuntimeError as error:
                # Sans caméra, le parcours continue quand même : on se contente d'un avertissement
                print(f"Warning: obstacle photo could not be captured: {error}")

            # LED allumé pendant 2s
            blink_leds(times=2)              

            # On attend tant que l'objet bloque le passage, le robot bipe et remesure la distance objet robot
            while read_ultrasonic_distance() < FAR_DISTANCE:
                sound_buzzer(times=3, duration=0.2, interval=0.2)
                time.sleep(OBSTACLE_POLL_INTERVAL)
    except KeyboardInterrupt:
        # déclenché par le bouton Interrupt de Jupyter
        print("Ending")
    finally:
        _close_systems()                     